In [ ]:
from datasets import load_dataset

# FEVER

In [2]:
fever = load_dataset("fever", 'v1.0')

In [ ]:
fever['train'][2]

In [3]:
wiki_pages = load_dataset("fever", "wiki_pages")

In [ ]:
x = 193183
wiki_pages['wikipedia_pages'][x:(x+1)]

## Match evidence from wiki_pages to FEVER

In [4]:
from tqdm import tqdm
import pandas as pd

### Dictionary for wikipedia evidence

In [5]:
wiki_lookup = {}
for item in wiki_pages['wikipedia_pages']:
    article_id = item['id']

    if not article_id:
        continue

    sentences = {}

    for l in item['lines'].split('\n'):
        parts = l.split('\t')
        # at least 2 parts: sentence_id and sentence
        if len(parts) >= 2:
            try:
                sentence_id = int(parts[0])
                sentence_text = parts[1]

                sentences[sentence_id] = sentence_text

            except ValueError:
                continue

    wiki_lookup[article_id] = sentences

## Training

### Matching claims with their evidence

In [ ]:
final_fever = []
dropped_count = 0

claim_only_no_evidence = []

for example in tqdm(fever['train'], total=len(fever["train"])):
    claim_id = example['id']
    claim = example['claim']
    label = example['label']

    evidence_article = example['evidence_wiki_url']
    evidence_sentence_id = example['evidence_sentence_id']

    evidence_text = None
    evidence_url = None


    if label != 'NOT ENOUGH INFO':
        
        if evidence_article and evidence_sentence_id is not None:
            article = wiki_lookup.get(evidence_article)
            if article and (evidence_sentence_id in article):
                evidence_text = article[evidence_sentence_id]
                evidence_url = f"https://en.wikipedia.org/wiki/{evidence_article}"

            else:
                # drop entry if evidence not found
                # our model will learn to predict on (claim + evidence)
                dropped_count += 1
                claim_only_no_evidence.append(example)
                continue

        else:
            dropped_count += 1
            continue    

    # daca labelul e NEI
    # For the first two classes (SUPPORTS and REFUTES), 
    # systems and annotators need to also return the combination of sentences 
    # forming the necessary evidence supporting or refuting the claim.
    # source: https://paperswithcode.com/dataset/fever
    else:
        evidence_url = None
        evidence_text = ""      


    verifiable = "NOT VERIFIABLE" if label == 'NOT ENOUGH INFO' else "VERIFIABLE"
    human_verified = "NO"


    final_fever.append({
        "id": claim_id,
        "claim": claim,
        "label": label,
        "claimURL": None,
        "evidence_url": evidence_url,
        "evidence_text": evidence_text,
        "human_verified": human_verified,
        "verifiable": verifiable,
    })

df_final = pd.DataFrame(final_fever)

### Treating claims labeled as SUPPORTED or REFUTED that have NO CLEAR EVIDENCE SENTENCE as not fit for training

In [ ]:
wiki_lookup.get('Adrienne_Bailon')

In [ ]:
claim_only_no_evidence[1]

In [ ]:
claim_only_no_evidence

In [ ]:
import gc
del wiki_pages
del fever
del final_fever
gc.collect()

In [ ]:
dropped_count

In [ ]:
df_final.shape[0]

In [ ]:
df_final.head(3)

In [ ]:
df_final

In [18]:
df_final.insert(0, 'unique_id', range(len(df_final)))

In [ ]:
df_final

In [ ]:
label_counts = df_final['label'].value_counts().reset_index()
label_counts.columns = ['label', 'count']

total_entries = df_final.shape[0]
label_counts['proportion'] = label_counts['count'] / total_entries

print('Statistics:')
print(label_counts)

In [21]:
import os

In [23]:
df_final.to_json(os.path.join(os.getcwd(), "prepped_fever_train.json"), orient="records", lines=True)


## Testing

In [ ]:
fever['paper_test'][0]

In [ ]:
test_fever = []
dropped_count = 0

claim_only_no_evidence = []

for example in tqdm(fever['paper_test'], total=len(fever["paper_test"])):
    claim_id = example['id']
    claim = example['claim']
    label = example['label']

    evidence_article = example['evidence_wiki_url']
    evidence_sentence_id = example['evidence_sentence_id']

    evidence_text = None
    evidence_url = None


    if label != 'NOT ENOUGH INFO':
        
        if evidence_article and evidence_sentence_id is not None:
            article = wiki_lookup.get(evidence_article)
            if article and (evidence_sentence_id in article):
                evidence_text = article[evidence_sentence_id]
                evidence_url = f"https://en.wikipedia.org/wiki/{evidence_article}"

            else:
                # drop entry if evidence not found
                # our model will learn to predict on (claim + evidence)
                dropped_count += 1
                claim_only_no_evidence.append(example)
                continue

        else:
            dropped_count += 1
            continue    

    # daca labelul e NEI
    # For the first two classes (SUPPORTS and REFUTES), 
    # systems and annotators need to also return the combination of sentences 
    # forming the necessary evidence supporting or refuting the claim.
    # source: https://paperswithcode.com/dataset/fever
    else:
        evidence_url = None
        evidence_text = ""      


    verifiable = "NOT VERIFIABLE" if label == 'NOT ENOUGH INFO' else "VERIFIABLE"
    human_verified = "NO"


    test_fever.append({
        "id": claim_id,
        "claim": claim,
        "label": label,
        "claimURL": None,
        "evidence_url": evidence_url,
        "evidence_text": evidence_text,
        "human_verified": human_verified,
        "verifiable": verifiable,
    })

df_final_test = pd.DataFrame(test_fever)

In [16]:
df_final_test.insert(0, 'unique_id', range(len(df_final_test)))

In [18]:
import os

In [19]:
df_final_test.to_json(os.path.join(os.getcwd(), "prepped_fever_test.json"), orient="records", lines=True)

## Validation

In [ ]:
fever['labelled_dev'][0]

In [ ]:
validation_fever = []
dropped_count = 0

claim_only_no_evidence = []

for example in tqdm(fever['labelled_dev'], total=len(fever["labelled_dev"])):
    claim_id = example['id']
    claim = example['claim']
    label = example['label']

    evidence_article = example['evidence_wiki_url']
    evidence_sentence_id = example['evidence_sentence_id']

    evidence_text = None
    evidence_url = None


    if label != 'NOT ENOUGH INFO':
        
        if evidence_article and evidence_sentence_id is not None:
            article = wiki_lookup.get(evidence_article)
            if article and (evidence_sentence_id in article):
                evidence_text = article[evidence_sentence_id]
                evidence_url = f"https://en.wikipedia.org/wiki/{evidence_article}"

            else:
                # drop entry if evidence not found
                # our model will learn to predict on (claim + evidence)
                dropped_count += 1
                claim_only_no_evidence.append(example)
                continue

        else:
            dropped_count += 1
            continue    

    # daca labelul e NEI
    # For the first two classes (SUPPORTS and REFUTES), 
    # systems and annotators need to also return the combination of sentences 
    # forming the necessary evidence supporting or refuting the claim.
    # source: https://paperswithcode.com/dataset/fever
    else:
        evidence_url = None
        evidence_text = ""      


    verifiable = "NOT VERIFIABLE" if label == 'NOT ENOUGH INFO' else "VERIFIABLE"
    human_verified = "NO"


    validation_fever.append({
        "id": claim_id,
        "claim": claim,
        "label": label,
        "claimURL": None,
        "evidence_url": evidence_url,
        "evidence_text": evidence_text,
        "human_verified": human_verified,
        "verifiable": verifiable,
    })

df_final_validation = pd.DataFrame(validation_fever)

In [8]:
df_final_validation.insert(0, 'unique_id', range(len(df_final_validation)))

In [ ]:
df_final_validation

In [12]:
import os

In [ ]:
df_final_validation.to_json(os.path.join(os.getcwd(), "prepped_fever_validation.json"), orient="records", lines=True)